## **Challenger model validation**

In [0]:
%pip install databricks-feature-engineering xgboost --quiet

In [0]:
from mlflow import MlflowClient
import mlflow
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pyspark.sql.functions as F


In [0]:
catalog_name = "ct_oil_gas" 
gold_schema_name = "sc_gold"
eval_split_table_name = "demand_eval_split"
feature_table_name = "daily_demand_features"
demand_labels_table_name = "daily_demand_labels"
# model_name = f"{catalog_name}.{gold_schema_name}.daily_oil_gas_demand_xg_model"
# model_name = "ct_oil_gas.sc_gold.daily_oil_gas_demand_xg_model"

RANDOM_STATE=  42

In [0]:
model_alias = "Challenger"

client  = MlflowClient()
model_name = "ct_oil_gas.sc_gold.daily_oil_gas_demand_xg_model"


challenger_model = client.get_model_version_by_alias(model_name,model_alias)
challenger_model_version = int(challenger_model.version)



In [0]:

challenger_model_version = int(challenger_model.version)

print(f"Validating {model_alias} model for {challenger_model_name} on model version {challenger_model_version}")


### Model performance metric

We want to validate the model performance metric. Typically, we want to compare this metric obtained for the Challenger model against that of the Champion model. Since we have yet to register a Champion model, we will only retrieve the metric for the Challenger model without doing a comparison.

The registered model captures information about the MLflow experiment run, where the model metrics were logged during training. This gives you traceability from the deployed model back to the initial training runs.

Here, we will use the F1 score for the out-of-sample test data set aside at training time.

In [0]:
challenger_model_run_id = challenger_model.run_id
challenger_val_r2 = mlflow.get_run(challenger_model_run_id).data.metrics['val_r2']

In [0]:
challenger_val_r2

In [0]:
def score_model_batch(
    catalog_name: str,
    gold_schema_name: str,
    demand_labels_table_name: str,
    model_name: str,
    model_alias: str,
    fe: FeatureEngineeringClient,
# -> tuple[DataFrame, float]
) :
    """Load the test split from the labels table and score it with the model.

    Returns the joined (prediction, label, features) DataFrame and the test R2.
    """
    labels_df = spark.table(f"{catalog_name}.{gold_schema_name}.{demand_labels_table_name}")
    test_labels_df = labels_df.filter("split == 'test'")
    test_eval_df = test_labels_df.select("p_key", "transaction_date")

    predictions_df = fe.score_batch(
        model_uri=f"models:/{model_name}@{model_alias}",
        df=test_eval_df,
    )

    prediction_with_label_df = predictions_df.select(
        "p_key", "transaction_date", "product_name", "destination_city", "prediction"
    ).join(
        test_labels_df.select("p_key", "transaction_date", "total_demand"),
        on=["p_key", "transaction_date"],
        how="inner",
    )

    n_expected = test_eval_df.count()
    n_actual = prediction_with_label_df.count()
    if n_actual != n_expected:
        raise ValueError(
            f"Row count mismatch after scoring/join: expected {n_expected}, got {n_actual}."
        )

    pdf = prediction_with_label_df.select("total_demand", "prediction").toPandas()
    test_r2 = r2_score(pdf["total_demand"], pdf["prediction"])

    return prediction_with_label_df, test_r2

In [0]:
fe = FeatureEngineeringClient()

In [0]:
challenger_predictions_df, challenger_test_r2 = score_model_batch(
    catalog_name=catalog_name,
    gold_schema_name=gold_schema_name,
    demand_labels_table_name=demand_labels_table_name,
    model_name=model_name,
    model_alias="Challenger",
    fe=fe,
)

print(f"Test R2: {challenger_test_r2}")
# display(predictions_df)

In [0]:

try:
    # compare the challenger R2 score to the existing champion if exists
    champion_model = client.get_model_version_by_alias(model_name,'Champion')
    champion_r2 =mlflow.get_run(champion_model.run_id).data.metrics['val_r2']
    print(f'Champion R2 score: {champion_r2}. Challenger R2 score: {challenger_val_r2}.')
    
    _, champion_test_r2 = score_model_batch(
    catalog_name=catalog_name,
    gold_schema_name=gold_schema_name,
    demand_labels_table_name=demand_labels_table_name,
    model_name=model_name,
    model_alias="Champion",
    fe=fe,
)
    metric_val_r2_passed = challenger_val_r2 > champion_r2
    metric_test_r2_passed = challenger_test_r2 > champion_test_r2
    print(f"Model {model_name} version {challenger_model.version} metric_val_r2_passed: {metric_val_r2_passed}")
    print(f"Model {model_name} version {challenger_model.version} metric_test_r2_passed: {metric_test_r2_passed}")

except:
    print("No champion found. Accept the Challenger model as its the first one")
    metric_val_r2_passed = True
    metric_test_r2_passed = True

print(f"Model {model_name} version {challenger_model.version} metric_val_r2_passed: {metric_val_r2_passed}")

client.set_model_version_tag(name=model_name, version=str(challenger_model.version),key='metric_val_r2_passed', value=metric_val_r2_passed)
client.set_model_version_tag(name=model_name, version=str(challenger_model.version),key='metric_test_r2_passed', value=metric_test_r2_passed)


In [0]:
results = client.get_model_version(model_name, challenger_model_version)
results.tags


In [0]:
if results.tags["metric_val_r2_passed"] == "True" and results.tags["metric_test_r2_passed"] == "True" :
  print('register model as Champion!')
  client.set_registered_model_alias(
    name=model_name,
    alias="Champion",
    version=challenger_model_version
  )
else:
  raise Exception("Model not ready for promotion")